In [ ]:
!pip install langchain-text-splitters

In [ ]:
!pip install bert-score

In [ ]:
import json
import re
from pathlib import Path

import tiktoken
from langchain_text_splitters import RecursiveCharacterTextSplitter

INPUT_MD = "output.md"
INPUT_MD_2 = "output2.md"
OUTPUT_JSON = "chunks.json"
CHUNK_OVERLAP = 153
CHUNK_SIZE = 1024 - CHUNK_OVERLAP
MIN_CHUNK_SIZE = 512 - CHUNK_OVERLAP
TOKENIZER = "cl100k_base"

RE_IMG = re.compile(r"!\[([^\]]*)\]\(([^)]+)\)")
RE_BLOCK_FORMULA = re.compile(r"\$\$(.+?)\$\$", re.DOTALL)
RE_INLINE_FORMULA = re.compile(r"(?<!\$)\$([^\$\n]+?)\$(?!\$)")
RE_HTML_TABLE = re.compile(r"<table\b.*?</table>", re.DOTALL | re.I)
RE_MD_TABLE = re.compile(r"(?:^\|[^\n]*\|\s*$\n?){2,}", re.M)
PH_RE = re.compile(r"(\[\[(?:FORMULA|TABLE|IMAGE)_\d+\]\])")
RE_SENT = re.compile(r"(?<=[.!?])\s+")
RE_DATE = re.compile(r"\b(?:\d{1,2}[./-]\d{1,2}[./-]\d{4}|\d{4}[./-]\d{1,2}[./-]\d{1,2})\b")
RE_NUMBER = re.compile(r"""(?<![\w.])[+-]?(?:\d{1,3}(?:[ _]\d{3})+|\d+)(?:[.,]\d+)?(?:[eE][+-]?\d+)?(?![\w.])""",re.VERBOSE)

enc = tiktoken.get_encoding(TOKENIZER)

FORMULAS = {}
TABLES = {}
IMAGES = {}
DATES = {}
NUMBERS={}
STORE = {}

def protect_pattern(text, pattern, store, prefix, block=False):
    def repl(m):
        key = f"[[{prefix}_{len(store):04d}]]"
        store[key] = m.group(0)
        return f"\n\n{key}\n\n" if block else key
    return pattern.sub(repl, text)


def protect_images(text):
    def repl(m):
        alt, src = m.group(1), m.group(2)
        key = f"[[IMAGE_{len(IMAGES):04d}]]"
        IMAGES[key] = {
            "alt": alt,
            "src": None if src.startswith("data:image") else src,
            "is_base64": src.startswith("data:image"),
            "src_len": len(src),
        }
        return key
    return RE_IMG.sub(repl, text)


def protect(text):
    text = protect_pattern(text, RE_BLOCK_FORMULA, FORMULAS, "FORMULA", block=True)
    text = protect_pattern(text, RE_HTML_TABLE, TABLES, "TABLE", block=True)
    text = protect_pattern(text, RE_MD_TABLE, TABLES, "TABLE", block=True)
    text = protect_pattern(text, RE_INLINE_FORMULA, FORMULAS, "FORMULA", block=False)
    text = protect_pattern(text, RE_DATE, DATES, "DATE", block=True)
    text = protect_pattern(text, RE_NUMBER, NUMBERS, "NUMBER", block=True)
    return text


def restore(text):
    for k, v in STORE.items():
        if k in text:
            text = text.replace(k, v)
    return text


def count(text):
    total = 0
    for part in PH_RE.split(text):
        if not part:
            continue
        value = STORE.get(part)
        total += len(enc.encode(value if isinstance(value, str) else part))
    return total

def _tail_for_overlap(text, max_tokens):
    sentences = RE_SENT.split(text)
    picked, total = [], 0
    for s in reversed(sentences):
        if PH_RE.search(s):
            continue
        s_tokens = count(s)
        if s_tokens > max_tokens:
            continue
        if picked and total + s_tokens > max_tokens:
            break
        picked.insert(0, s)
        total += s_tokens
        if total >= max_tokens:
            break
    return " ".join(picked)

def add_overlap(chunks, overlap_tokens, chunk_size):
    if len(chunks) < 2:
        return chunks
    result = [chunks[0]]
    for i in range(1, len(chunks)):
        budget = chunk_size - count(chunks[i])
        budget = max(0, min(budget, overlap_tokens))
        tail = _tail_for_overlap(chunks[i-1], budget) if budget > 0 else ""
        result.append(f"{tail}\n\n{chunks[i]}" if tail else chunks[i])
    return result

def merge_small(chunks):
    if not chunks:
        return []
    merged, buf = [], ""
    for c in chunks:
        cand = f"{buf}\n\n{c}".strip() if buf else c
        if count(cand) <= CHUNK_SIZE:
            buf = cand
        else:
            if buf:
                merged.append(buf)
            buf = c
    if buf:
        merged.append(buf)

    i = 0
    while i < len(merged):
        if count(merged[i]) >= MIN_CHUNK_SIZE:
            i += 1
            continue
        done = False
        if i > 0 and count(merged[i - 1] + "\n\n" + merged[i]) <= CHUNK_SIZE:
            merged[i - 1] += "\n\n" + merged[i]
            merged.pop(i)
            done = True
        if not done and i < len(merged) - 1 and count(merged[i] + "\n\n" + merged[i + 1]) <= CHUNK_SIZE:
            merged[i + 1] = merged[i] + "\n\n" + merged[i + 1]
            merged.pop(i)
            done = True
        if not done:
            i += 1
    return merged


def protect_text(text):
    FORMULAS.clear()
    TABLES.clear()
    IMAGES.clear()
    STORE.clear()
    NUMBERS.clear()
    DATES.clear()

    text = protect_images(text)
    protected = protect(text)

    STORE.update(FORMULAS)
    STORE.update(TABLES)
    STORE.update(DATES)
    STORE.update(NUMBERS)
    return protected
    # splitter = RecursiveCharacterTextSplitter(
    #     chunk_size=CHUNK_SIZE,
    #     chunk_overlap=0,
    #     length_function=count,
    #     separators=["\n\n", "\n", " ", ""],
    #     keep_separator=True,
    # )
    # pieces = splitter.split_text(protected)
    # pieces = merge_small(pieces)
    # pieces = add_overlap(pieces, CHUNK_OVERLAP, CHUNK_SIZE)

    # chunks = []
    # for i, p in enumerate(pieces):
    #     restored = restore(p)
    #     img_refs = [{"placeholder": k, **IMAGES[k]} for k in IMAGES if k in p]
    #     chunks.append({
    #         "text": restored,
    #         "masked_text": p,
    #         "metadata": {
    #             "index": i,
    #             "tokens": len(enc.encode(restored)),
    #             "images": img_refs,
    #             "has_formula": any(k in p for k in FORMULAS),
    #             "has_table": any(k in p for k in TABLES),
    #             "has_image": bool(img_refs),
    #             "n_atomic": len(PH_RE.findall(p)),
    #         },
    #     })
    #return chunks

In [ ]:
text1 = Path(INPUT_MD).read_text(encoding="utf-8")
text2 = Path(INPUT_MD_2).read_text(encoding="utf-8")


In [ ]:
protected = protect_text(text1+text2)

# Path(OUTPUT_JSON).write_text(
#     json.dumps(chunks, ensure_ascii=False, indent=2),
#     encoding="utf-8",
# )

In [ ]:
STORE

In [ ]:
# sizes = [c["metadata"]["tokens"] for c in chunks]
# big = [c for c in chunks if c["metadata"]["tokens"] > 1024]
# print(f"Кол-во чанков: {len(chunks)}")
# print(f"Размеры: min={min(sizes)}, avg={sum(sizes)//len(sizes)}, max={max(sizes)}")
# print(f"Кол-во формул: {len(FORMULAS)} | Кол-во таблиц: {len(TABLES)} | Кол-во картинок: {len(IMAGES)}")
# print(f"Чанков с формулой: {sum(c['metadata']['has_formula'] for c in chunks)}, "
#       f"с таблицей: {sum(c['metadata']['has_table'] for c in chunks)}, "
#       f"с картинкой: {sum(c['metadata']['has_image'] for c in chunks)}")
# print(f"Чанков, превышающих лимит ({1024}): {len(big)}")

### Очистка
- Непечатные и управляющие символы (ASCII < 32, кроме \n, \r, \t)
- Лишние пробелы и табуляции
- HTML/XML-теги
- Стоп-слова (с осторожностью для технических текстов)
- Персональные данные (GDPR, ФЗ-152)
- Удаление картинок

In [ ]:
import re

CTRL_RE      = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]")
INVISIBLE_RE = re.compile(r"[\u200b-\u200f\u2028-\u202f\u2060\ufeff\u00ad]")
SPACEY_RE    = re.compile(r"[\u00a0\u1680\u2000-\u200a\u202f\u205f\u3000]")
MULTISPACE_RE = re.compile(r"[ ]{2,}")
SPACES_NL_RE = re.compile(r"[ \t]+\n|\n[ \t]+")
SPACES_NL_CLEAN_RE = re.compile(r"[ \t]*\n[ \t]*")
MULTINL_RE    = re.compile(r"\n{3,}")
CR_RE         = re.compile(r"\r")
IMG_MASK = re.compile(r"\[\[IMAGE_\d{4}\]\]")

PII_PATTERNS = [
    re.compile(r"\b[\w.+-]+@[\w-]+\.[\w.-]+\b"),                              # email
    re.compile(r"\b(?:\+7|8)[\s\-\(]?\d{3}[\s\-\)]?\d{3}[\s\-]?\d{2}[\s\-]?\d{2}\b"),  # ru phone
]


def mask_pii(text):
    for pat in PII_PATTERNS:
        text = pat.sub("[PII]", text)
    return text

def count_issues(text):
    ctrl_hits       = CTRL_RE.findall(text)
    invisible_hits  = INVISIBLE_RE.findall(text)
    nbsp_hits       = SPACEY_RE.findall(text)
    multispace_hits = MULTISPACE_RE.findall(text)
    spaces_nl_hits  = SPACES_NL_RE.findall(text)
    multinl_hits    = MULTINL_RE.findall(text)
    pii_pattern1 = PII_PATTERNS[0].findall(text)
    pii_pattern2 = PII_PATTERNS[1].findall(text)

    return {
        "Управляющие символы ASCII": len(ctrl_hits),
        "Невидимые юникод-символы": len(invisible_hits),
        "Юникод-пробелы": len(nbsp_hits),
        "2+ подряд идущих пробелов": len(multispace_hits),
        "Пробелы перед/после перевода строки (висячие пробелы)": len(spaces_nl_hits),
        "3+ подряд идущих перевода строки": len(multinl_hits),
        "ФЗ-152": len(pii_pattern1) + len(pii_pattern2),
    }


def clean_chunk(text):
    text = IMG_MASK.sub("", text)
    text = CTRL_RE.sub("", text)
    text = INVISIBLE_RE.sub("", text)
    text = SPACEY_RE.sub(" ", text)
    text = MULTISPACE_RE.sub(" ", text)
    text = SPACES_NL_CLEAN_RE.sub("\n", text)
    text = MULTINL_RE.sub("\n\n", text)
    for pat in PII_PATTERNS:
        text = pat.sub(" ", text)
    return text.strip()

def clean(chunks):
    return [clean_chunk(c["masked_text"]) for c in chunks]

In [ ]:
for k, v in count_issues(protected).items():
    print(f"  {k:12}: {v}")
print()

In [ ]:
clean_protected = clean_chunk(protected) # has ALL masks

In [ ]:
for k, v in count_issues(clean_protected).items():
    print(f"  {k:12}: {v}")
print()

In [ ]:
from bert_score import score

P, R, F1 = score(
    cands=[protected],
    refs=[clean_protected],
    lang="ru",
    verbose=False,
)
print(f"BERTScore F1 = {F1.item():.4f}")
print(f"BERTScore Precision = {P.item():.4f}")
print(f"BERTScore Recall = {R.item():.4f}")

### Нормализация

Рекомендация: нормализация дат, единиц измерения, формул, таблиц в одном стиле

Нормализация единиц измерения: приведение к СИ через Pint.
Нормализация дат: → ISO 8601 (2025-03-12) через dateutil.
Нормализация чисел: единый формат, научная нотация → десятичная.
Нормализация формул: LaTeX → канонический вид через SymPy, сохранение AST.
Нормализация таблиц: единая структура, типы данных через pandas.

In [ ]:
pip install markdown

In [ ]:
import markdown


def markdown_table_to_html(text: str) -> str:
    return markdown.markdown(
        text,
        extensions=["tables"]
    )
def convert_markdown_tables_in_store(store, tables_dict, debug=False):
    ok = skipped = failed = 0
    for key in tables_dict:
        value = store.get(key)
        if not isinstance(value, str):
            skipped += 1
            continue

        v = value.lstrip()

        if v.startswith("<"):
            skipped += 1
            continue
        if not v.startswith("|"):
            skipped += 1
            continue

        try:
            html = markdown_table_to_html(value)
            if html and html.strip():
                store[key] = html.strip()
                ok += 1
            else:
                failed += 1
                if debug:
                    print(f"{key}: markdown вернул пусто")
        except Exception as e:
            failed += 1
            if debug:
                print(f"{key}: {type(e).__name__}: {e}")

In [ ]:
convert_markdown_tables_in_store(STORE, TABLES, debug=True)

In [ ]:
STORE

In [ ]:
!pip install sympy

In [ ]:
!pip install "antlr4-python3-runtime==4.11"

In [ ]:
import re

def preprocess_formula(store,formulas_store):
    def normalize(text):
        text = text.strip()
        text = re.sub(r"^\${1,2}|\${1,2}$", "", text)
        text = re.sub(r"[\u00A0\u1680\u2000-\u200A\u202F\u205F\u3000]", " ", text)
        text = re.sub(r"\\(?:,|;|:|!| )", "", text)
        text = text.replace("~", "")
        text = re.sub(r"\s+", " ", text)
        text = re.sub(r"\s*([{}_^])\s*", r"\1", text)
        text = re.sub(r"(?<=\d)\s+(?=\d)", "", text)
        text = re.sub(r"\\([a-zA-Z]+)\s*\{", r"\\\1{", text)
        text = re.sub(r"\{\s*([^{}]*?)\s*\}", r"{\1}", text)
        return text.strip()

    for key in formulas_store:
        store[key] = normalize(formulas_store[key])


In [ ]:
preprocess_formula(STORE, FORMULAS)

In [ ]:
from dateutil import parser


def normalize_dates(store, dates_store) -> tuple[str, int]:
    def replace_date(value):
        try:
            if re.match(r"^\d{1,2}[./-]\d{1,2}[./-]\d{4}$", value):
                date = parser.parse(value, dayfirst=True)
            else:
                date = parser.parse(value)
            return date.strftime("%Y-%m-%d")
        except (ValueError, OverflowError):
            return value

    for key in dates_store:
        store[key] = replace_date(dates_store[key])

In [ ]:
normalize_dates(STORE, DATES)

In [ ]:
from decimal import Decimal


def normalize_numbers(store, numbers_store) -> tuple[str, int]:
    def replace_number(value):
        value = value.replace(" ", "").replace("_", "")
        value = value.replace(",", ".")

        try:
            number = Decimal(value)
            result = format(number, "f")
            if "." in result:
                result = result.rstrip("0").rstrip(".")

            if result in ("-0", "+0"):
                result = "0"
            return result
        except Exception:
            pass

    for key in numbers_store:
        store[key] = replace_number(numbers_store[key])

In [ ]:
normalize_numbers(STORE, NUMBERS)

In [ ]:
STORE

In [ ]:
Path("clean_protected.md").write_text(clean_protected, encoding="utf-8")

In [ ]:
import pandas as pd
from pathlib import Path

def inspect_parquet(path: str, n_examples: int = 2):
    """Читает parquet, печатает инфо о колонках и примеры строк."""
    path = Path(path)
    if not path.exists():
        print(f"Файл не найден: {path}")
        return None

    df = pd.read_parquet(path)

    print("=" * 70)
    print(f"Файл: {path.name}")
    print(f"Размер: {path.stat().st_size / 1024:.1f} KB")
    print(f"Строк: {len(df)}, Колонок: {len(df.columns)}")
    print("=" * 70)

    # ─── колонки ───
    print("\nКОЛОНКИ:")
    print(f"{'#':>3}  {'имя':<30} {'dtype':<20} {'non-null':>10}  пример")
    print("-" * 100)

    for i, col in enumerate(df.columns):
        dtype = str(df[col].dtype)
        non_null = df[col].notna().sum()

        # первый непустой пример
        sample = df[col].dropna()
        if len(sample):
            val = sample.iloc[0]
            # обрезаем длинные значения
            if isinstance(val, (list, tuple)):
                sample_str = f"[{type(val).__name__} len={len(val)}]"
            elif isinstance(val, str):
                sample_str = val[:40].replace("\n", " ")
                if len(val) > 40:
                    sample_str += "..."
            else:
                sample_str = str(val)[:40]
        else:
            sample_str = "(пусто)"

        print(f"{i:>3}  {col:<30} {dtype:<20} {non_null:>10}  {sample_str}")

    # ─── примеры строк ───
    print(f"\nПРИМЕРЫ СТРОК (первые {n_examples}):")
    print("=" * 70)

    for idx in range(min(n_examples, len(df))):
        row = df.iloc[idx]
        print(f"\n--- Строка #{idx} ---")
        for col in df.columns:
            val = row[col]

            # форматирование для читаемости
            if isinstance(val, (list, tuple)):
                # для списков — показываем длину и первые элементы
                n = len(val)
                if n and isinstance(val[0], (int, float)):
                    display = f"[{n} элементов] {val[:5]}..."
                elif n and isinstance(val[0], (list, tuple)):
                    display = f"[{n} векторов] первый: {val[0][:5]}..."
                else:
                    display = f"[{n} элементов] {val[:3]}"
            elif isinstance(val, str):
                if len(val) > 200:
                    display = val[:200] + "..."
                else:
                    display = val
            else:
                display = val

            print(f"  {col}: {display}")

    print()
    return df


DEFAULT = "text_units.parquet"

path = DEFAULT
inspect_parquet(path, n_examples=2)

In [ ]:
import pandas as pd
from pathlib import Path

TEXT_UNITS = "text_units.parquet"


def replace_markers(text: str, store: dict) -> str:
    """Заменяет все маркеры [[TYPE_NNNN]] на значения из STORE."""
    for key, value in store.items():
        if key in text:
            text = text.replace(key, value)
    return text


def restore_text_units(path: str, store: dict, keep_masked: bool = True):
    """
    Читает text_units.parquet, заменяет маркеры на значения из STORE.
    Если keep_masked=True — сохраняет исходный текст в колонку text_masked.
    """
    df = pd.read_parquet(path)

    print(f"Чанков до замены: {len(df)}")
    print(f"Колонки: {list(df.columns)}")

    # сохраняем masked-версию
    if keep_masked:
        df["text_masked"] = df["text"]

    # считаем сколько маркеров было
    def count_markers(t):
        import re
        return len(re.findall(r"\[\[(?:FORMULA|TABLE|DATE|NUMBER)_\d{4}\]\]", t))

    before = df["text"].apply(count_markers).sum()

    # заменяем
    df["text"] = df["text"].apply(lambda t: replace_markers(t, store))

    # считаем сколько осталось
    after = df["text"].apply(count_markers).sum()

    print(f"Маркеров до замены: {before}")
    print(f"Маркеров после:     {after}")

    # сохраняем
    #df.to_parquet(path)

    print(f"Сохранено: {path}")
    print()
    print("Пример первого чанка до:")
    print(df.iloc[5]["text_masked"][:200] if keep_masked else "—")
    print()
    print("Пример первого чанка после:")
    print(df.iloc[5]["text"][:200])

    return df


# ─── использование ───
# STORE — ваш словарь {[[FORMULA_0000]]: "...", [[TABLE_0000]]: "..."}
restore_text_units(TEXT_UNITS, STORE, keep_masked=True)

In [ ]:
# Final: restore tables and formulas
clean_restored_chunks = restore(clean_chunk)

### Токенизация (Оценка)

Рекомендлация: по предложениям дальше по словам

In [27]:
import re
import pandas as pd
from collections import defaultdict, Counter


TEXT_UNITS = "text_units.parquet"

MARKER_RE = re.compile(r"\[\[(FORMULA|TABLE|DATE|NUMBER|IMAGE)_\d{4}\]\]")
SENT_RE = re.compile(r"(?<=[.!?])\s+(?=[А-ЯЁA-Z«\(\[]|\d)")

WORD_RE = re.compile(
    r"\[\[(?:FORMULA|TABLE|DATE|NUMBER|IMAGE)_\d{4}\]\]"
    r"|[А-Яа-яЁёA-Za-z]+(?:[-/][А-Яа-яЁёA-Za-z]+)*"
    r"|\d+(?:[.,]\d+)?",
    re.UNICODE,
)


def split_sentences(text):
    return [s.strip() for s in SENT_RE.split(text) if s.strip()]


def split_words(sentence):
    return WORD_RE.findall(sentence)


def pipeline(text):
    sentences = split_sentences(text)
    words = []
    for s in sentences:
        words.extend(split_words(s))
    return sentences, words


def marker_word_metrics(texts):
    stats = defaultdict(lambda: {"count": 0, "whole": 0, "split": 0, "examples": []})

    for text in texts:
        _, words = pipeline(text)
        word_set = set(words)

        for m in MARKER_RE.finditer(text):
            key = m.group(0)
            kind = m.group(1)
            stats[kind]["count"] += 1
            if key in word_set:
                stats[kind]["whole"] += 1
            else:
                stats[kind]["split"] += 1
                if len(stats[kind]["examples"]) < 3:
                    stats[kind]["examples"].append(key)

    return dict(stats)


def build_vocab(texts):
    counter = Counter()
    for text in texts:
        _, words = pipeline(text)
        for w in words:
            if not MARKER_RE.fullmatch(w):
                counter[w.lower()] += 1
    return {w for w, c in counter.items() if c >= 2}


def oov_rate(texts, vocab):
    total = oov = 0
    oov_words = Counter()
    for text in texts:
        _, words = pipeline(text)
        for w in words:
            if MARKER_RE.fullmatch(w):
                continue
            total += 1
            if w.lower() not in vocab:
                oov += 1
                oov_words[w] += 1
    return {"total": total, "oov": oov,
            "rate": oov / total if total else 0.0,
            "top": oov_words.most_common(10)}


def report(path):
    df = pd.read_parquet(path)
    texts = df["text"].tolist()

    total_sents = sum(len(split_sentences(t)) for t in texts)
    total_words = sum(len(pipeline(t)[1]) for t in texts)

    print(f"Чанков: {len(texts)}")
    print(f"Предложений: {total_sents}")
    print(f"Слов: {total_words}")
    print()

    print("── МАРКЕРЫ (по словам) ──")
    mm = marker_word_metrics(texts)
    for kind in ["FORMULA", "TABLE", "DATE", "NUMBER", "IMAGE"]:
        if kind not in mm:
            continue
        v = mm[kind]
        pct = v["whole"] / v["count"] * 100 if v["count"] else 0
        print(f"  {kind:<8} всего={v['count']} "
              f"целых_слов={v['whole']} ({pct:.1f}%) "
              f"разбито={v['split']}")
        for ex in v["examples"]:
            print(f"           пример разрыва: {ex}")

    print()
    print("── OOV RATE ──")
    vocab = build_vocab(texts)
    ov = oov_rate(texts, vocab)
    print(f"  словарь: {len(vocab)} слов")
    print(f"  слов всего={ov['total']} "
          f"oov={ov['oov']} "
          f"rate={ov['rate']*100:.2f}%")
    if ov["top"]:
        print(f"  топ OOV: {ov['top']}")


report(TEXT_UNITS)

Чанков: 28
Предложений: 820
Слов: 12174

── МАРКЕРЫ (по словам) ──
  FORMULA  всего=44 целых_слов=44 (100.0%) разбито=0
  TABLE    всего=14 целых_слов=14 (100.0%) разбито=0
  DATE     всего=1 целых_слов=1 (100.0%) разбито=0
  NUMBER   всего=441 целых_слов=441 (100.0%) разбито=0

── OOV RATE ──
  словарь: 1545 слов
  слов всего=11674 oov=1360 rate=11.65%
  топ OOV: [('Сергей', 1), ('Валериевич', 1), ('кафедре', 1), ('металловедения', 1), ('физики', 1), ('профессор', 1), ('Филиппов', 1), ('Кантор', 1), ('ИМЕТ', 1), ('РАН', 1)]


In [45]:
import re
import pandas as pd
from collections import defaultdict, Counter
from transformers import AutoTokenizer


TEXT_UNITS = "text_units.parquet"
MODEL_NAME = "cointegrated/rubert-tiny2"

MARKER_RE = re.compile(r"\[\[(FORMULA|TABLE|DATE|NUMBER|IMAGE)_(\d{4})\]\]")
SHORT_RE = re.compile(r"<([ФТДЧК])(\d+)>")
WORD_RE = re.compile(r"[А-Яа-яЁёA-Za-z]+(?:[-/][А-Яа-яЁёA-Za-z]+)*", re.UNICODE)

RU_SHORT = {
    "FORMULA": "Ф",
    "TABLE":   "Т",
    "DATE":    "Д",
    "NUMBER":  "Ч",
    "IMAGE":   "К",
}

RU_BACK = {v: k for k, v in RU_SHORT.items()}

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def replace_short(text):
    def repl(m):
        kind, num = m.group(1), m.group(2)
        return f"<{RU_SHORT[kind]}{int(num)}>"
    return MARKER_RE.sub(repl, text)


def marker_token_metrics(texts):
    stats = defaultdict(lambda: {"count": 0, "tokens": [], "examples": []})

    for text in texts:
        for m in SHORT_RE.finditer(text):
            key = m.group(0)
            kind = RU_BACK[m.group(1)]
            pieces = tokenizer.tokenize(key)
            stats[kind]["count"] += 1
            stats[kind]["tokens"].append(len(pieces))
            if len(stats[kind]["examples"]) < 3:
                stats[kind]["examples"].append((key, pieces))

    return {
        k: {
            "count":     v["count"],
            "single":    sum(1 for t in v["tokens"] if t == 1),
            "avg":       sum(v["tokens"]) / len(v["tokens"]) if v["tokens"] else 0,
            "max":       max(v["tokens"]) if v["tokens"] else 0,
            "examples":  v["examples"],
        }
        for k, v in stats.items()
    }


def oov_rate(texts, threshold=3):
    total = oov = 0
    oov_words = Counter()

    for text in texts:
        clean = SHORT_RE.sub(" ", text)
        for w in WORD_RE.findall(clean):
            total += 1
            if len(tokenizer.tokenize(w)) >= threshold:
                oov += 1
                oov_words[w] += 1

    return {
        "total": total,
        "oov": oov,
        "rate": oov / total if total else 0.0,
        "top": oov_words.most_common(15),
    }


def report(texts, label):
    print(f"═══ {label} ═══")
    print(f"Модель: {MODEL_NAME}")
    print(f"Чанков: {len(texts)}\n")

    print("── МАРКЕРЫ ──")
    mm = marker_token_metrics(texts)
    for kind in ["FORMULA", "TABLE", "DATE", "NUMBER", "IMAGE"]:
        if kind not in mm:
            continue
        v = mm[kind]
        pct = v["single"] / v["count"] * 100 if v["count"] else 0
        print(f"  {kind:<8} всего={v['count']} "
              f"1_токен={v['single']} ({pct:.1f}%) "
              f"avg={v['avg']:.2f} max={v['max']}")
        for key, pieces in v["examples"]:
            print(f"           пример: {key} → {pieces}")

    print()
    print("── OOV RATE (>= 3 субтокенов) ──")
    ov = oov_rate(texts)
    print(f"  слов всего={ov['total']} "
          f"oov={ov['oov']} "
          f"rate={ov['rate']*100:.2f}%")
    if ov["top"]:
        print(f"  топ OOV: {ov['top']}")
    print()


df = pd.read_parquet(TEXT_UNITS)
texts = df["text"].tolist()

before = texts
after = [replace_short(t) for t in texts]

print("Пример замены:")
print(f"  было:  {before[1][:200]}")
print(f"  стало: {after[1][:200]}")
print()

report(before, "ЛАТИНСКИЕ МАРКЕРЫ [[FORMULA_0000]]")
report(after,  "КОРОТКИЕ КИРИЛЛИЧЕСКИЕ <Ф0>")

Пример замены:
  было:   температур испытаний [[FORMULA_0001]]

Массовые измерения геометрии изломов ударных образцов выявили их информативные параметры: уширение ударного образца в месте выбега трещины, линейные размеры рас
  стало:  температур испытаний <Ф1>

Массовые измерения геометрии изломов ударных образцов выявили их информативные параметры: уширение ударного образца в месте выбега трещины, линейные размеры расслоев «в пла

═══ ЛАТИНСКИЕ МАРКЕРЫ [[FORMULA_0000]] ═══
Модель: cointegrated/rubert-tiny2
Чанков: 28

── МАРКЕРЫ ──

── OOV RATE (>= 3 субтокенов) ──
  слов всего=11583 oov=1462 rate=12.62%
  топ OOV: [('NUMBER', 444), ('FORMULA', 44), ('микроструктуры', 38), ('Скородумов', 24), ('изломов', 24), ('неоднородности', 23), ('МИСиС', 22), ('НМФА', 19), ('микроструктур', 19), ('излома', 14), ('TABLE', 14), ('Пышминцев', 14), ('трещиностойкости', 13), ('надрезом', 12), ('неметаллических', 11)]

═══ КОРОТКИЕ КИРИЛЛИЧЕСКИЕ <Ф0> ═══
Модель: cointegrated/rubert-tiny2
Чанков: 28

──

### Векторизация (e5)

In [32]:
import lancedb
import numpy as np

db = lancedb.connect("/Users/alisaegorova/PycharmProjects/Graphs&GraphModels/Avtoreferat_Skorodumov_S.V./lancedb")

for table_name in ["entity_description", "text_unit_text", "community_full_content"]:
    df = db.open_table(table_name).to_pandas()
    vectors = np.vstack(df["vector"].tolist())
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    vectors_norm = vectors / (norms + 1e-12)

    sim = vectors_norm @ vectors_norm.T
    n = len(vectors)
    mask = ~np.eye(n, dtype=bool)

    print(f"── {table_name} ──")
    print(f"  строк: {n}, размерность: {vectors.shape[1]}")
    print(f"  косинус: min={sim[mask].min():.3f} "
          f"avg={sim[mask].mean():.3f} "
          f"max={sim[mask].max():.3f}")
    print()

── entity_description ──
  строк: 220, размерность: 768
  косинус: min=0.338 avg=0.603 max=0.990

── text_unit_text ──
  строк: 11, размерность: 768
  косинус: min=0.715 avg=0.807 max=0.902

── community_full_content ──
  строк: 3, размерность: 768
  косинус: min=0.734 avg=0.783 max=0.866



In [ ]:
# Final: restore tables and formulas
clean_restored_chunks = [restore(clean_chunk) for clean_chunk in clean_chunks]